<a href="https://colab.research.google.com/github/Ronglawan/PROJECT_PHY_483/blob/main/MINI_PROJECT_PHY_483.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.ensemble import RandomForestClassifier

from tensorflow import keras
from tensorflow.keras import layers

#Load dataset

In [ ]:
df = pd.read_csv("ev_battery_qc_data_2026_kaggle.csv")

print(df.head())

print(df.info())

print(df.describe())

In [ ]:
print(df.columns)

##Data preprocessing

In [ ]:
# ลบคอลัมน์ที่ไม่จำเป็น + กัน data leakage
df = df.drop(['Cell_ID','Batch_ID','Inspector_Comment','Defect_Type'], axis=1)

# จัดการ missing
df = df.dropna()

# Encode
le = LabelEncoder()
df['Production_Line'] = le.fit_transform(df['Production_Line'])
df['Shift'] = le.fit_transform(df['Shift'])
df['Supplier'] = le.fit_transform(df['Supplier'])
df['QC_Grade'] = le.fit_transform(df['QC_Grade'])

##Split data

In [ ]:
X = df.drop('QC_Grade', axis=1)
y = df['QC_Grade']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

##Normalize data

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

#Method
##Model 1: Deep Learning

In [ ]:
model = keras.Sequential([
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(16, activation='relu'),
    layers.Dense(3, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2
)

##Evaluate DL

In [ ]:
y_pred_dl = model.predict(X_test)
y_pred_dl = np.argmax(y_pred_dl, axis=1)

print("Deep Learning Accuracy:", accuracy_score(y_test, y_pred_dl))

##Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred_dl)
sns.heatmap(cm, annot=True, fmt='d')
plt.title("Deep Learning Confusion Matrix")
plt.show()

##Model 2: Random Forest

##Train/Test

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

##Train Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\n=== Random Forest ===")
print(classification_report(y_test, y_pred_rf))

##Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm_rf = confusion_matrix(y_test, y_pred_rf)

sns.heatmap(cm_rf, annot=True, fmt='d')
plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
labels = ['Grade A','Grade B','Scrap']
pred_labels = [labels[i] for i in y_pred_rf]

for i in range(10):
    print("Actual:", y_test.iloc[i], "| Predicted:", pred_labels[i])

##เปรียบเทียบผล

In [ ]:
print("\n=== Deep Learning ===")
print(classification_report(y_test, y_pred_dl))

print("\n=== Random Forest ===")
print(classification_report(y_test, y_pred_rf))

##Accuracy Graph

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.legend(['Train','Validation'])
plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.show()

##Loss Graph

In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.legend(['Train','Validation'])
plt.title("Model Loss")
plt.show()

In [ ]:
import pandas as pd

importance = rf.feature_importances_
features = X.columns

imp_df = pd.DataFrame({
    'Feature': features,
    'Importance': importance
}).sort_values(by='Importance', ascending=False)

print(imp_df)

In [ ]:
labels = ['Grade A','Grade B','Scrap']

pred_rf = [labels[i] for i in y_pred_rf]
pred_dl = [labels[i] for i in y_pred_dl]

for i in range(10):
    print("Actual:", y_test.iloc[i],
          "| RF:", pred_rf[i],
          "| DL:", pred_dl[i])

In [ ]:
import pandas as pd

labels = ['Grade A','Grade B','Scrap']

pred_rf = [labels[i] for i in y_pred_rf]
pred_dl = [labels[i] for i in y_pred_dl]

df_compare = pd.DataFrame({
    'Actual': y_test.values[:10],
    'Random Forest': pred_rf[:10],
    'Deep Learning': pred_dl[:10]
})

print(df_compare)